Extraemos archivo jsonl que contiene la data cruda

#**🧪 PRUEBA TÉCNICA – INGENIERO DE DATOS**
**Empresa**: PUNTORED  
**Objetivo**: Disposición final de datos, modelo analitico.  
**Fecha creación**: 29/03/2025  
**Autor**: Jorge Andres Mosquera  
**Version**: 1.0   
**Fecha modificación**:  
**Motivo modificación**:  

Lectura de tablas creadas en capa silver

In [0]:
display(dbutils.fs.ls(silver_path + "/transactions"))

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7520437425227480>, line 1
----> 1 display(dbutils.fs.ls(silver_path + "/transactions"))

NameError: name 'silver_path' is not defined

Lectura de tablas Silver

In [0]:
# Definición de tablas UC en Silver
transactions_table = "jorge_catalog.silver.transactions"
users_table        = "jorge_catalog.silver.users"
details_table      = "jorge_catalog.silver.transaction_details"

# Lectura desde Silver
df_tx      = spark.table(transactions_table)
df_users   = spark.table(users_table)
df_details = spark.table(details_table)

# Validación
print("Transactions:", df_tx.count())
print("Users:", df_users.count())
print("Details:", df_details.count())

Transactions: 50000
Users: 50000
Details: 50000


Calculo de métricas por usuario

In [0]:
from pyspark.sql.functions import count, sum, col, when

# Definir tabla destino en Gold
user_metrics_table = "jorge_catalog.gold.user_metrics"

# Calcular métricas por usuario
df_user_metrics = df_tx.groupBy("user_id").agg(
    count("*").alias("total_transactions"),
    sum("amount").alias("total_amount"),
    (sum(when(col("status") == "approved", 1).otherwise(0)) / count("*")).alias("success_rate")
)

# Guardar en Gold como tabla UC
df_user_metrics.write.format("delta").mode("overwrite").saveAsTable(user_metrics_table)

# Validación
print("User Metrics:", spark.table(user_metrics_table).count())

User Metrics: 40


Métricas por método de pago

In [0]:
from pyspark.sql.functions import count

response_metrics_table = "jorge_catalog.gold.response_metrics"

df_response_metrics = df_details.groupBy("response_code").agg(
    count("*").alias("total_transactions")
)

df_response_metrics.write.format("delta").mode("overwrite").saveAsTable(response_metrics_table)

print("Response Metrics:", spark.table(response_metrics_table).count())

Response Metrics: 10


Métricas por canal

In [0]:
from pyspark.sql.functions import count

# Definir tabla destino en Gold
response_metrics_table = "jorge_catalog.gold.response_metrics"

# Calcular métricas por código de respuesta
df_response_metrics = df_details.groupBy("response_code").agg(
    count("*").alias("total_transactions")
)

# Guardar en Gold como tabla UC
df_response_metrics.write.format("delta").mode("overwrite").saveAsTable(response_metrics_table)

# Validación
print("Response Metrics:", spark.table(response_metrics_table).count())

Response Metrics: 10


Crear vista en memoria

In [0]:
df_user_metrics.write.format("delta").mode("overwrite").saveAsTable("jorge_catalog.transacciones.gold_user_metrics")

Ejecucion consulta sql

In [0]:
%sql
SELECT user_id, total_transactions, total_amount, success_rate
FROM jorge_catalog.gold.user_metrics
ORDER BY total_amount DESC

user_id,total_transactions,total_amount,success_rate
Laura Jimenez,1314,73657356999,0.863013698630137
Daniela Soto,1291,70298021037,0.8497288923315259
Jose Aguilar,1250,67630530045,0.8392
Valentina Ortiz,1272,64057731835,0.8482704402515723
Felipe Romero,1302,62682758402,0.836405529953917
Roberto Vargas,1258,62642079978,0.8473767885532592
Luis Fernandez,1239,62474166612,0.8547215496368039
Natalia Nunez,1242,61964916725,0.8582930756843801
Emma Castillo,1330,61521669311,0.8383458646616542
Carlos Gomez,1274,61387699657,0.8547880690737834
